# 7. Prompt Based Similarity Sorting

*Goal: Sort data based off of similarity to user provided prompts.

## Step 1 — Import Libraries

Import the required modules:
- `pandas` for our DataFrame operations. (I like to import with `import pandas as pd` as it a fairly common pattern)
- `numpy` for similarity operations. (I like to import with `import numpy as np` as it a fairly common pattern)
- `SecretStr` from `pydantic`, which is a utility that prevents API keys from accidentally being printed to the console or saved in notebook output.

In [ ]:
import pandas as pd  # <--- DataFrame usage for dataset operations
from pydantic import SecretStr  # <--- Masked string
import numpy as np  # <--- Numerical operations for similarity calculations

## Step 2 — Load the Dataset

Read the Parquet file produced by notebook **4-cluster-data** into a Pandas DataFrame.

> `_04_clustered_commands.parquet` should contain the `CommandLine`, `Embedding`, `Cluser_DBSCAN`, and `Cluster_KMeans` columns.

In [ ]:
dataframe = pd.read_parquet("_04_clustered_commands.parquet")

## Step 3 — Initialize the Embeddings Client
Create an embeddings client that matches the method used to generate the embeddingss and assign it to the `embeddings_client` variable. The nice thing about using LangChain is that it makes it trivial to adjust AI providers. Depending on the provider you are using, import the appropriate Chat class.


- OpenAI `from langchain_openai import OpenAIEmbeddings`
- Google AI Studio `from langchain_google_genai import GoogleGenerativeAIEmbeddings`
- Ollama `from langchain_ollama import OllamaEmbeddings`

Both OpenAI and Google will require API keys, but if you are running Ollama locally, you will not need an API key. Here are examples of creating a chat client for these providers.

```python
OpenAIEmbeddings(
    model="text-embedding-3-large",
    api_key=SecretStr(api_key)
)
```

```python
GoogleGenerativeAIEmbeddings(
    model="gemini-embedding-001",
    api_key=SecretStr(api_key)
)
```

```python
OllamaEmbeddings(
    model="embeddinggemma"
)
```

Related Docs:
- https://docs.langchain.com/oss/python/integrations/embeddings/openai
- https://docs.langchain.com/oss/python/integrations/embeddings/google_generative_ai
- https://docs.langchain.com/oss/python/integrations/embeddings/ollama


In [ ]:
from langchain_openai import OpenAIEmbeddings

api_key = input("Enter your Azure OpenAI API key: ")

embeddings_client = OpenAIEmbeddings(
    model="text-embedding-3-large",
    api_key=SecretStr(api_key)
)

In [ ]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings

api_key = input("Enter your Google AI Studio API key: ")

embeddings_client = GoogleGenerativeAIEmbeddings(
    model="gemini-embedding-001",
    api_key=SecretStr(api_key)
)

In [ ]:
from langchain_ollama import OllamaEmbeddings

embeddings_client = OllamaEmbeddings(
    model="embeddinggemma"
)

## Step 4 - Create an Embedding Query
Set a `query_embedding` variable to an embedding vector that we can use for a simliarity query to sort our data with. Call the embedding client's `embed_query()` function and pass it a prompt.

Prompt example: `"interactions with volume shadows, vshadow, vssadmin, shadow copies"`

Related Docs:
- https://docs.langchain.com/oss/python/integrations/embeddings/openai#embed-single-texts
- https://docs.langchain.com/oss/python/integrations/embeddings/google_generative_ai#instantiation
- https://docs.langchain.com/oss/python/integrations/embeddings/ollama#embed-single-texts

In [ ]:
query_embedding = embeddings_client.embed_query("Volume shadow; vshadow; vssadmin; shadow copy; not cmd;")

## Step 5 - Sort Data

Compute a **cosine similarity score** between the query embedding and every command embedding in the dataset, then sort by that score descending.

1. Stack all stored embeddings into a 2-D matrix (one row per command).
2. Reshape the query vector to a single-row matrix so scikit-learn's `cosine_similarity` can compare it against every row at once.
3. Extract the resulting score array (index `[0]`) and assign it as a new column on the DataFrame.
4. Sort descending — the most semantically similar commands appear first.


Import the `cosine_similarity` function from the `sklearn.metrics.pairwise` module.

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

### Build the Embedding Matrix

Convert the DataFrame that we imported in step 2s `Embedding` column into a single 2-D NumPy array so each row represents one command embedding and each column represents an embedding dimension.

Use `np.vstack(...)` to stack all embedding vectors from `dataframe["Embedding"].values` into a matrix and store it in a variable named `embedding_matrix`.

This prepares the data for vectorized cosine similarity against the query embedding in the next step.

In [ ]:
# Stack each embedding list into a 2-D NumPy matrix (rows = commands, cols = embedding dimensions)
embedding_matrix = np.vstack(dataframe["Embedding"].values)

### Reshape the Query Vector

Convert `query_embedding` (a plain Python list) into a NumPy array and reshape it to a single-row 2-D matrix so that scikit-learn's `cosine_similarity` function can accept it as input — it requires both arguments to be 2-D.

Use `np.array(query_embedding).reshape(1, -1)` and store the result in a variable named `query_vector`.

- `1` means one row (our single query)
- `-1` tells NumPy to infer the number of columns from the embedding length automatically

In [ ]:
# Reshape query to (1, dimensions) — cosine_similarity expects 2-D inputs
query_vector = np.array(query_embedding).reshape(1, -1)


### Compute Cosine Similarity Scores

Use `cosine_similarity(query_vector, embedding_matrix)` to compute a similarity score between the query and every command in the dataset. This returns a 2-D array with shape `(1, n_commands)` — index `[0]` to extract the flat 1-D score array.

Assign the scores as a new column on the DataFrame:

```python
dataframe["SimilarityScore"] = cosine_similarity(query_vector, embedding_matrix)[0]
```

In [ ]:
# Compute cosine similarity between the query and every command; extract the flat score array ([0])
dataframe["SimilarityScore"] = cosine_similarity(query_vector, embedding_matrix)[0]

### Sort by Similarity Score

Sort the DataFrame by `SimilarityScore` in descending order so the most semantically similar commands appear first, and store the result in a new variable named `dataframe_sorted`.

The DataFrame's `.sort_values()` function can do this. Pass it the column name and sort descending.

In [ ]:

# Sort descending so the most semantically similar commands appear first
dataframe_sorted = dataframe.sort_values("SimilarityScore", ascending=False)


### Preview the Results

Display the top 20 most similar commands by selecting only the `CommandLine` and `SimilarityScore` columns from `dataframe_sorted` and calling `.head(20)` on the result.

In [ ]:
dataframe_sorted[["CommandLine", "SimilarityScore"]].head(20)

**Exercise 7.1: Experiment**

Play around with using different prompts and note how it sorts the data. See if you can tweak prompts to better sort the data noting what changed.

Export the data to a csv file if it makes exploring the sorted data easier for you.
